# STRATA: `junta` com o SHA-256 do shard num fio só dele

Na versão anterior (seção 31.2), o fio principal calculava o SHA-256 do shard **e** gravava, enquanto um fio auxiliar calculava o SHA-256 de cada tensor sobre os mesmos bytes. No 8B do Colab, a decodificação levou 119 s e o hash do shard 122 s: o hash era a fila mais lenta.

Agora:

- o SHA-256 do shard (o que se compara com o oid LFS do Hub) ganhou **um fio só dele**, separado da gravação;
- o SHA-256 de cada tensor, que cobria os mesmos bytes, roda só com `--tudo`.

**Regra de sucesso, escrita antes de medir:** no 8B, `junta` novo pelo menos 25% mais rápido que o anterior na mesma máquina, com os 4 shards conferindo com o SHA-256 do Hub. Se não chegar a isso, a mudança não vira padrão.

**Runtime:** CPU com RAM alta (~25–35 min, ~50 GB de disco). No fim, a última célula baixa um zip; mande ele na thread.


In [ ]:
# 1. A máquina (o SHA-256 depende muito de o processador ter a extensão SHA)
import os, shutil, subprocess
def sh(cmd, mostra=True):
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, executable="/bin/bash")
    if mostra:
        print(r.stdout[-4000:])
    if r.returncode:
        raise SystemExit(f"falhou: {cmd}")
    return r.stdout
PASTA = "/content/strata_ab"
os.makedirs(PASTA, exist_ok=True)
cpu = sh("lscpu | grep -E 'Model name|^CPU\\(s\\)|Thread'", False)
flags = open("/proc/cpuinfo").read()
info = f"{cpu.strip()}\nsha_ni: {'sha_ni' in flags} · avx2: {'avx2' in flags}\nRAM {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9:.1f} GB · disco livre {shutil.disk_usage(PASTA).free / 1e9:.0f} GB"
print(info)

In [ ]:
# 2. Rust e as duas versões do strata-ref (antiga = a da seção 31.2) (≈ 5 min)
RAMO, ANTIGO = "claude/project-thread-btiyii", "dfec6f0"
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
REPO, VELHO = "/content/strata", "/content/strata_antigo"
if not os.path.exists(REPO):
    sh(f"git clone -q --filter=blob:none -b {RAMO} https://github.com/cantyll/strata {REPO}")
if not os.path.exists(VELHO):
    sh(f"cd {REPO} && git worktree add -q {VELHO} {ANTIGO}")
sh(f"cd {REPO} && git log -1 --oneline && cd {VELHO} && git log -1 --oneline")
for r in (REPO, VELHO):
    sh(f"cd {r}/leitor-rs && cargo build --release -q 2>&1 | tail -3; ls -la target/release/strata-ref")
sh("pip install -q -U huggingface_hub hf_xet psutil", False)
NOVO_BIN, VELHO_BIN = f"{REPO}/leitor-rs/target/release/strata-ref", f"{VELHO}/leitor-rs/target/release/strata-ref"

In [ ]:
# 3. O teste completo no 8B com a versão nova: baixa, grava os estratos e volta (≈ 15 min)
import sys
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
cmd = [sys.executable, "-u", "tests/dist/teste_grande.py", "--hf", "unsloth/Meta-Llama-3.1-8B-Instruct",
       "--gguf-repo", "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF", "--gguf", "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
       "--dir", PASTA, "--bin", NOVO_BIN]
p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end="")
p.wait()
assert p.returncode == 0, "o teste falhou"

In [ ]:
# 4. As duas versões do junta sobre os mesmos arquivos (≈ 15–25 min)
import time, re
IDX = f"{PASTA}/dist/model.strata.index.json"
GG = f"{PASTA}/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
def limpa_cache():
    subprocess.run("sync; echo 3 > /proc/sys/vm/drop_caches", shell=True, stderr=subprocess.DEVNULL)
def junta(bin_, extra, nome):
    limpa_cache()
    t = time.time()
    out = sh(f"{bin_} junta {IDX} {extra} --base {GG} 2>&1", False)
    s = time.time() - t
    open(f"{PASTA}/resultados/ab_{nome}.txt", "w").write(out)
    ok = len(re.findall(r"\(confere\)", out))
    assert ok == 4, out[-2000:]
    print(f"{nome}: {s:.0f} s, {ok} shards conferidos\n  " + "\n  ".join(l for l in out.splitlines() if l.startswith(("ok:", "tempos:"))))
    return s
# duas rodadas alternadas, para não confundir a mudança com a variação da máquina
tempos = {"anterior": [], "novo": []}
for r in range(2):
    tempos["anterior"].append(junta(VELHO_BIN, "", f"anterior_{r}"))
    tempos["novo"].append(junta(NOVO_BIN, "", f"novo_{r}"))
# uma volta de verdade, gravando os shards, com cada versão
SAIDA = f"{PASTA}/saida"
for nome, b in (("anterior", VELHO_BIN), ("novo", NOVO_BIN)):
    os.makedirs(SAIDA, exist_ok=True)
    tempos[f"{nome}, gravando"] = [junta(b, SAIDA, f"{nome}_gravando")]
    shutil.rmtree(SAIDA)

In [ ]:
# 5. Resumo e download
med = {k: sorted(v)[len(v) // 2] if len(v) % 2 else sum(sorted(v)[len(v) // 2 - 1:len(v) // 2 + 1]) / 2 for k, v in tempos.items()}
linhas = ["# junta: SHA do shard num fio só dele (Llama-3.1-8B, Colab)", "", "```", info, "```", "", "| versão | rodadas (s) | mediana (s) |", "|---|---|---:|"]
linhas += [f"| {k} | {' / '.join(f'{x:.0f}' for x in v)} | {med[k]:.0f} |" for k, v in tempos.items()]
g = med["anterior"] / med["novo"]
linhas += ["", f"Sem gravar: anterior / novo = **{g:.2f}×** ({(1 - med['novo'] / med['anterior']) * 100:.0f}% mais rápido). Regra: pelo menos 25% → **{'passou' if med['novo'] <= 0.75 * med['anterior'] else 'não passou'}**.",
           f"Gravando: {med['anterior, gravando'] / med['novo, gravando']:.2f}×."]
txt = "\n".join(linhas)
open(f"{PASTA}/resultados/ab2.md", "w").write(txt + "\n")
from IPython.display import Markdown, display
display(Markdown(txt))
z = shutil.make_archive("/content/strata_junta_ab2", "zip", f"{PASTA}/resultados")
try:
    from google.colab import files
    files.download(z)
except Exception:
    print(z)
